# Task 4｜实验四：结构化处理与日期抽取

## 1. 实验目的

学习将客服消息中的订单编号、退款金额、日期表达和请求类型转换为标准化字段。

## 2. 实验内容

1. 使用 JEV 判断用户请求类型和日期表达类型。
2. 使用 Python 提取订单编号、金额和日期信息。
3. 将相对日期转换为标准日期。
4. 校验字段完整性，并识别需要人工确认的情况。

## 3. 核心原则

模型负责理解和判断，程序负责解析、校验和标准化。
提取结果不等于事实已经核实，含糊的日期和缺失的字段不能直接用于业务执行。

In [1]:
import re
from datetime import date, datetime, timedelta

from typesafe_sdk import TypeSafeClient, Choice, Noul

client = TypeSafeClient(model="jev-1.13.0")

# 固定参考日期，方便重复实验并核对结果
REFERENCE_DATE = date(2026, 10, 20)

print("参考日期：", REFERENCE_DATE)
print("JEV 客户端初始化完成")

参考日期： 2026-10-20
JEV 客户端初始化完成


In [2]:
TEST_CASES = [
    {
        "id": "T1",
        "message": "请在下周三之前把订单 A104 的 56 元退款处理好。"
    },
    {
        "id": "T2",
        "message": "订单 B207 的 120 元退款请在三天内处理。"
    },
    {
        "id": "T3",
        "message": "订单 C308 的 99.5 元退款请在10月25日前处理。"
    },
    {
        "id": "T4",
        "message": "订单 D409 的退款请尽快处理。"
    },
    {
        "id": "T5",
        "message": "订单 E510 的退款请在2027-01-05前处理。"
    }
]

print("测试案例数量：", len(TEST_CASES))

for case in TEST_CASES:
    print(f'{case["id"]}: {case["message"]}')

测试案例数量： 5
T1: 请在下周三之前把订单 A104 的 56 元退款处理好。
T2: 订单 B207 的 120 元退款请在三天内处理。
T3: 订单 C308 的 99.5 元退款请在10月25日前处理。
T4: 订单 D409 的退款请尽快处理。
T5: 订单 E510 的退款请在2027-01-05前处理。


In [3]:
# 第一步：定义 JEV 的判断问题

REQUEST_QUESTIONS = {
    "request_type": Choice(
        instructions="判断用户的主要请求类型，只能选择一个。",
        criteria={
            "refund_processing": "要求处理退款或催促退款",
            "order_status": "询问订单状态或物流信息",
            "refund_policy": "询问退款政策或退款规则",
            "complaint": "投诉或表达不满",
            "other": "不属于以上类型"
        }
    ),

    "date_type": Choice(
        instructions="判断用户消息中的时间表达类型，只能选择一个。",
        criteria={
            "relative_weekday": "下周三、下周一等相对星期日期",
            "relative_duration": "三天内、一周内等时间间隔",
            "explicit_chinese_date": "10月25日、11月3日等中文日期",
            "explicit_iso_date": "2027-01-05等标准数字日期",
            "no_specific_date": "尽快、越快越好等没有明确日期的表达",
            "ambiguous": "时间表达含糊，无法明确判断"
        }
    ),

    "has_deadline": Noul(
        instructions="用户是否提出了明确的日期或时间期限？有明确期限返回True，没有明确期限返回False。"
    )
}

print("JEV 判断问题定义完成")

JEV 判断问题定义完成


In [4]:
# 第二步：调用 JEV，并保存判断结果

JEV_RECORDS = []

for case in TEST_CASES:
    result = client.system_one(
        state={
            "message": case["message"]
        },
        questions=REQUEST_QUESTIONS
    )

    answers = result.answers

    record = {
        "id": case["id"],
        "message": case["message"],
        "request_type": answers["request_type"].choice,
        "request_confidence": answers["request_type"].confidence,
        "date_type": answers["date_type"].choice,
        "date_confidence": answers["date_type"].confidence,
        "has_deadline": answers["has_deadline"].noul
    }

    JEV_RECORDS.append(record)

    print("=" * 50)
    print("案例：", record["id"])
    print("用户消息：", record["message"])
    print("请求类型：", record["request_type"])
    print("请求判断置信度：", round(record["request_confidence"], 3))
    print("日期类型：", record["date_type"])
    print("日期判断置信度：", round(record["date_confidence"], 3))
    print("是否有明确期限：", record["has_deadline"])

案例： T1
用户消息： 请在下周三之前把订单 A104 的 56 元退款处理好。
请求类型： refund_processing
请求判断置信度： 1.0
日期类型： relative_weekday
日期判断置信度： 1.0
是否有明确期限： 0.96
案例： T2
用户消息： 订单 B207 的 120 元退款请在三天内处理。
请求类型： refund_processing
请求判断置信度： 1.0
日期类型： relative_duration
日期判断置信度： 1.0
是否有明确期限： 0.95
案例： T3
用户消息： 订单 C308 的 99.5 元退款请在10月25日前处理。
请求类型： refund_processing
请求判断置信度： 1.0
日期类型： explicit_chinese_date
日期判断置信度： 1.0
是否有明确期限： 0.99
案例： T4
用户消息： 订单 D409 的退款请尽快处理。
请求类型： refund_processing
请求判断置信度： 1.0
日期类型： no_specific_date
日期判断置信度： 1.0
是否有明确期限： 0.1
案例： T5
用户消息： 订单 E510 的退款请在2027-01-05前处理。
请求类型： refund_processing
请求判断置信度： 1.0
日期类型： explicit_iso_date
日期判断置信度： 1.0
是否有明确期限： 0.99


In [5]:
import re

def extract_order_id(message):
    """从消息中提取订单号。"""
    match = re.search(r"订单\s*([A-Z]\d+)", message)
    return match.group(1) if match else None


def extract_amount(message):
    """从消息中提取金额。"""
    match = re.search(r"(\d+(?:\.\d+)?)\s*元", message)
    return float(match.group(1)) if match else None


print("===== 字段提取测试 =====")

for case in TEST_CASES:
    message = case["message"]

    print(
        case["id"],
        "订单号：", extract_order_id(message),
        "金额：", extract_amount(message)
    )

===== 字段提取测试 =====
T1 订单号： A104 金额： 56.0
T2 订单号： B207 金额： 120.0
T3 订单号： C308 金额： 99.5
T4 订单号： D409 金额： None
T5 订单号： E510 金额： None


In [6]:
from datetime import date, timedelta
import re

WEEKDAY_CN = {
    "一": 0, "二": 1, "三": 2, "四": 3,
    "五": 4, "六": 5, "日": 6, "天": 6
}

CN_NUMBERS = {
    "一": 1, "二": 2, "三": 3, "四": 4,
    "五": 5, "六": 6, "七": 7, "八": 8,
    "九": 9, "十": 10
}


def normalize_deadline(record):
    message = record["message"]
    date_type = record["date_type"]
    confidence = record["date_confidence"]
    deadline_probability = float(record["has_deadline"])

    # 1. 先检查模型判断是否可靠
    if date_type == "no_specific_date":
        return None, "未提供具体日期，保留原始表达"

    if date_type == "ambiguous":
        return None, "日期表达含糊，需要人工确认"

    if deadline_probability < 0.80:
        return None, "期限判断不够明确，需要人工确认"

    if confidence < 0.75:
        return None, "日期类型判断置信度不足，需要人工确认"

    # 2. 解析“下周三”等相对星期表达
    if date_type == "relative_weekday":
        match = re.search(r"下周([一二三四五六日天])", message)

        if not match:
            return None, "无法提取星期，需要人工确认"

        weekday = WEEKDAY_CN[match.group(1)]

        # 找到参考日期所在周的下周一
        next_week_monday = (
            REFERENCE_DATE
            - timedelta(days=REFERENCE_DATE.weekday())
            + timedelta(days=7)
        )

        deadline = next_week_monday + timedelta(days=weekday)

        return deadline, "相对星期日期已标准化"

    # 3. 解析“三天内”等时间间隔
    if date_type == "relative_duration":
        match = re.search(
            r"(\d+|[一二三四五六七八九十]+)天内",
            message
        )

        if not match:
            return None, "无法提取时间间隔，需要人工确认"

        number_text = match.group(1)

        if number_text.isdigit():
            days = int(number_text)
        elif number_text in CN_NUMBERS:
            days = CN_NUMBERS[number_text]
        else:
            return None, "暂不支持该数字表达，需要人工确认"

        deadline = REFERENCE_DATE + timedelta(days=days)

        return deadline, "相对时间间隔已标准化"

    # 4. 解析“10月25日”等中文日期
    if date_type == "explicit_chinese_date":
        match = re.search(r"(\d{1,2})月(\d{1,2})日?", message)

        if not match:
            return None, "无法提取中文日期，需要人工确认"

        month, day = map(int, match.groups())
        year = REFERENCE_DATE.year

        try:
            deadline = date(year, month, day)

            # 没有写年份且日期已经过去，暂按下一年解释
            if deadline < REFERENCE_DATE:
                deadline = date(year + 1, month, day)

        except ValueError:
            return None, "日期无效，需要人工确认"

        return deadline, "中文日期已标准化"

    # 5. 解析“2027-01-05”等标准日期
    if date_type == "explicit_iso_date":
        match = re.search(
            r"\d{4}-\d{2}-\d{2}",
            message
        )

        if not match:
            return None, "无法提取标准日期，需要人工确认"

        try:
            deadline = date.fromisoformat(match.group())
        except ValueError:
            return None, "日期格式无效，需要人工确认"

        return deadline, "标准日期已解析"

    return None, "暂不支持该日期类型，需要人工确认"

In [7]:
jev_by_id = {
    record["id"]: record
    for record in JEV_RECORDS
}

STRUCTURED_RECORDS = []

for case in TEST_CASES:
    record = jev_by_id[case["id"]]

    deadline, date_status = normalize_deadline(record)

    structured = {
        "id": case["id"],
        "request_type": record["request_type"],
        "order_id": extract_order_id(case["message"]),
        "amount_yuan": extract_amount(case["message"]),
        "date_type": record["date_type"],
        "deadline_probability": round(
            float(record["has_deadline"]), 3
        ),
        "deadline_date": (
            deadline.isoformat()
            if deadline is not None
            else None
        ),
        "date_status": date_status
    }

    STRUCTURED_RECORDS.append(structured)

    print("=" * 45)
    for key, value in structured.items():
        print(f"{key}: {value}")

id: T1
request_type: refund_processing
order_id: A104
amount_yuan: 56.0
date_type: relative_weekday
deadline_probability: 0.96
deadline_date: 2026-10-28
date_status: 相对星期日期已标准化
id: T2
request_type: refund_processing
order_id: B207
amount_yuan: 120.0
date_type: relative_duration
deadline_probability: 0.95
deadline_date: 2026-10-23
date_status: 相对时间间隔已标准化
id: T3
request_type: refund_processing
order_id: C308
amount_yuan: 99.5
date_type: explicit_chinese_date
deadline_probability: 0.99
deadline_date: 2026-10-25
date_status: 中文日期已标准化
id: T4
request_type: refund_processing
order_id: D409
amount_yuan: None
date_type: no_specific_date
deadline_probability: 0.1
deadline_date: None
date_status: 未提供具体日期，保留原始表达
id: T5
request_type: refund_processing
order_id: E510
amount_yuan: None
date_type: explicit_iso_date
deadline_probability: 0.99
deadline_date: 2027-01-05
date_status: 标准日期已解析


In [9]:
# 第八步：校验结构化结果，识别异常字段


def validate_record(record):
    warnings = []

    # 1. 检查订单号
    if not record["order_id"]:
        warnings.append("缺少订单号，无法定位订单")

    # 2. 检查金额
    amount = record["amount_yuan"]

    if amount is None:
        warnings.append("缺少退款金额，禁止自动发起退款")
    elif amount <= 0:
        warnings.append("退款金额必须大于 0")

    # 3. 检查日期类型与实际日期
    date_type = record["date_type"]
    deadline = record["deadline_date"]
    score = record["deadline_probability"]

    if date_type == "no_specific_date":
        warnings.append("没有明确日期，保留原始表达")

        # 没有明确日期，却认为存在明确期限，可能发生了判断冲突
        if score >= 0.80:
            warnings.append("日期类型与期限判断不一致，需要人工确认")

    elif date_type == "ambiguous":
        warnings.append("日期表达含糊，需要人工确认")

    else:
        # 模型识别出了具体日期类型，但 Python 没有解析出日期
        if deadline is None:
            warnings.append("日期解析失败，需要人工确认")

        # 具体日期类型与期限判断分数不一致
        if score < 0.80:
            warnings.append("日期类型与期限判断可能冲突，需要人工确认")

    return warnings


print("===== 修正后的结构化结果校验 =====")

for record in STRUCTURED_RECORDS:
    warnings = validate_record(record)

    print(f"\n案例：{record['id']}")

    if warnings:
        for warning in warnings:
            print("⚠️", warning)
    else:
        print("基础字段校验通过")


===== 修正后的结构化结果校验 =====

案例：T1
基础字段校验通过

案例：T2
基础字段校验通过

案例：T3
基础字段校验通过

案例：T4
⚠️ 缺少退款金额，禁止自动发起退款
⚠️ 没有明确日期，保留原始表达

案例：T5
⚠️ 缺少退款金额，禁止自动发起退款


### 第九步：边界测试与异常处理

**实验目的：** 检查结构化数据校验程序在异常输入下能否正常工作，并识别需要人工确认的情况。

**测试场景：**
1. 缺少订单号。
2. 缺少退款金额。
3. 退款金额为负数。
4. 日期类型明确，但日期解析失败。
5. 日期类型与期限判断分数存在矛盾。
6. 日期表达含糊。

**预期结果：** 程序能够识别异常数据并给出警告，而不是直接将其用于业务处理。

**核心原则：** 数据提取成功不代表数据有效，数据有效也不代表业务操作已经获得授权。

In [10]:

# 第九步：边界测试与异常处理

BOUNDARY_CASES = [
    {
        "id": "B1",
        "order_id": "",
        "amount_yuan": 56.0,
        "date_type": "explicit_iso_date",
        "deadline_date": date(2027, 1, 5),
        "deadline_probability": 0.99
    },
    {
        "id": "B2",
        "order_id": "A201",
        "amount_yuan": None,
        "date_type": "explicit_iso_date",
        "deadline_date": date(2027, 1, 5),
        "deadline_probability": 0.99
    },
    {
        "id": "B3",
        "order_id": "A202",
        "amount_yuan": -10.0,
        "date_type": "explicit_iso_date",
        "deadline_date": date(2027, 1, 5),
        "deadline_probability": 0.99
    },
    {
        "id": "B4",
        "order_id": "A203",
        "amount_yuan": 56.0,
        "date_type": "explicit_iso_date",
        "deadline_date": None,
        "deadline_probability": 0.99
    },
    {
        "id": "B5",
        "order_id": "A204",
        "amount_yuan": 56.0,
        "date_type": "no_specific_date",
        "deadline_date": None,
        "deadline_probability": 0.95
    },
    {
        "id": "B6",
        "order_id": "A205",
        "amount_yuan": 56.0,
        "date_type": "ambiguous",
        "deadline_date": None,
        "deadline_probability": 0.40
    }
]

print("===== 边界测试结果 =====")

for record in BOUNDARY_CASES:
    warnings = validate_record(record)

    print(f"\n案例：{record['id']}")

    if warnings:
        for warning in warnings:
            print("⚠️", warning)
    else:
        print("基础字段校验通过")


===== 边界测试结果 =====

案例：B1
⚠️ 缺少订单号，无法定位订单

案例：B2
⚠️ 缺少退款金额，禁止自动发起退款

案例：B3
⚠️ 退款金额必须大于 0

案例：B4
⚠️ 日期解析失败，需要人工确认

案例：B5
⚠️ 没有明确日期，保留原始表达
⚠️ 日期类型与期限判断不一致，需要人工确认

案例：B6
⚠️ 日期表达含糊，需要人工确认


## 实验四：日期抽取与结构化处理——学习心得

### 1. 实验目的

本实验以客服退款工单为场景，探索如何将自然语言中的订单号、退款金额和处理期限等信息提取为结构化字段，并通过 Python 程序完成日期标准化、字段校验和异常拦截。

### 2. 实验内容

1. 使用 JEV 对用户消息进行请求类型、日期类型和期限信息判断。
2. 使用 Python 正则表达式提取订单号与金额。
3. 根据固定参考日期，将相对日期和明确日期转换为标准日期。
4. 对结构化结果进行校验，识别缺失字段、无效金额、日期解析失败和判断结果不一致等问题。
5. 通过边界测试验证程序面对异常输入时能否正确发出警告。

### 3. 实验发现

- **模型负责理解，程序负责标准化。** 模型能够识别“下周三”“三天内”等自然语言表达，但仍需要程序将其转换成具体日期。
- **提取成功不等于校验通过。** 即使订单号和金额已经提取，也需要检查字段是否完整、金额是否合理。
- **置信度不能代替逻辑校验。** 当日期类型与期限判断分数不一致时，即使模型给出了较高分数，也应该触发检查。
- **异常情况需要有明确的处理路径。** 对于缺失金额、日期含糊或解析失败等情况，程序应保留原始表达并要求人工确认，而不是直接执行退款。
- **测试用例可以帮助发现规则漏洞。** 正常案例只能证明基本流程能够运行，边界测试才能暴露缺失字段和异常数据处理方面的问题。

### 4. 实验总结

通过本次实验，我初步理解了大模型结构化处理的基本流程：自然语言理解、字段提取、格式标准化、规则校验和异常处理。大模型输出的结构化结果不能直接视为已经核实的业务事实，还需要通过程序检查其完整性、合理性和一致性。

后续如果将该方案应用于真实客服系统，还需要进一步测试日期解析准确率、置信度阈值的有效性，以及异常输入和真实业务数据下的系统稳定性。